# Symmetric halo-orbit differential correction

Demonstrate the existing single-shooting corrector using the Earth–Moon northern halo reference from the targeting tests.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from tno_dynamics.propagation import propagate_cr3bp
from tno_dynamics.targeting import correct_halo_orbit

## Initial perturbation

Perturb only the two correction controls, $x_0$ and $\dot{y}_0$, while retaining the reference value of $z_0$.

In [ ]:
mu = 0.01215058560962404
reference_state = np.array(
    [-0.41456184803140111, 0.0, 0.90753120433295065, 0.0, 1.4076145460136695, 0.0]
)
state_guess = reference_state.copy()
state_guess[0] += 1.0e-6
state_guess[4] -= 1.0e-6

print("Reference state:", reference_state)
print("Perturbed initial guess:", state_guess)
print("Initial perturbation:", state_guess - reference_state)

## Differential-correction result

Correct $x_0$ and $\dot{y}_0$ so that $\dot{x}$ and $\dot{z}$ vanish at the downward symmetry-plane crossing.

In [ ]:
corrected_state, half_period, residual_history = correct_halo_orbit(state_guess, mu)
period = 2.0 * half_period

print("Corrected state:", corrected_state)
print("Correction relative to guess:", corrected_state - state_guess)
print(f"Half-period: {half_period:.16f}")
print(f"Full period: {period:.16f}")
print(f"Correction iterations: {residual_history.size}")
print(f"Initial residual: {residual_history[0]:.6e}")
print(f"Final residual: {residual_history[-1]:.6e}")

## Residual convergence

Track the Euclidean symmetry-condition residual at each correction iteration.

In [ ]:
iterations = np.arange(1, residual_history.size + 1)

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(iterations, residual_history, marker="o")
ax.set_xlabel("Correction iteration")
ax.set_ylabel("Residual norm")
ax.set_title("Halo differential-correction convergence")
ax.set_xticks(iterations)
ax.grid(True, which="both", alpha=0.3)
plt.show()

## Full-period closure

Propagate the corrected initial state over one complete period, measure closure, and inspect the orbit geometry.

In [ ]:
t_eval = np.linspace(0.0, period, 2001)
solution = propagate_cr3bp(corrected_state, (0.0, period), mu, t_eval=t_eval)
final_state = solution.y[:, -1]
closure = final_state - corrected_state

print("Full-period closure vector:", closure)
print(f"Euclidean closure norm: {np.linalg.norm(closure):.6e}")

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot(solution.y[0], solution.y[1], solution.y[2], label="Corrected halo orbit")
ax.scatter(*corrected_state[:3], marker="x", s=70, label="Initial state")
ax.scatter(-mu, 0.0, 0.0, s=90, label="Larger primary")
ax.scatter(1.0 - mu, 0.0, 0.0, s=50, label="Smaller primary")
ax.set_xlabel("Rotating $x$ coordinate")
ax.set_ylabel("Rotating $y$ coordinate")
ax.set_zlabel("Rotating $z$ coordinate")
ax.set_title("Corrected Earth–Moon northern halo orbit")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(solution.y[0], solution.y[2], label="Corrected halo orbit")
ax.scatter(corrected_state[0], corrected_state[2], marker="x", s=70, label="Initial state")
ax.scatter(-mu, 0.0, s=90, label="Larger primary")
ax.scatter(1.0 - mu, 0.0, s=50, label="Smaller primary")
ax.set_xlabel("Rotating $x$ coordinate")
ax.set_ylabel("Rotating $z$ coordinate")
ax.set_title("Halo orbit $x$–$z$ projection")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()